<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/2-Advance/FinRL_Ensemble_StockTrading_ICAIF_2020.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Deep Reinforcement Learning for Stock Trading from Scratch: Multiple Stock Trading Using Ensemble Strategy

Tutorials to use OpenAI DRL to trade multiple stocks using ensemble strategy in one Jupyter Notebook | Presented at ICAIF 2020

* This notebook is the reimplementation of our paper: Deep Reinforcement Learning for Automated Stock Trading: An Ensemble Strategy, using FinRL.
* Check out medium blog for detailed explanations: https://medium.com/@ai4finance/deep-reinforcement-learning-for-automated-stock-trading-f1dad0126a02
* Please report any issues to our Github: https://github.com/AI4Finance-LLC/FinRL-Library/issues
* **Pytorch Version**



# Content

* [1. Problem Definition](#0)
* [2. Getting Started - Load Python packages](#1)
    * [2.1. Install Packages](#1.1)    
    * [2.2. Check Additional Packages](#1.2)
    * [2.3. Import Packages](#1.3)
    * [2.4. Create Folders](#1.4)
* [3. Download Data](#2)
* [4. Preprocess Data](#3)        
    * [4.1. Technical Indicators](#3.1)
    * [4.2. Perform Feature Engineering](#3.2)
* [5.Build Environment](#4)  
    * [5.1. Training & Trade Data Split](#4.1)
    * [5.2. User-defined Environment](#4.2)   
    * [5.3. Initialize Environment](#4.3)    
* [6.Implement DRL Algorithms](#5)  
* [7.Backtesting Performance](#6)  
    * [7.1. BackTestStats](#6.1)
    * [7.2. BackTestPlot](#6.2)   
    * [7.3. Baseline Stats](#6.3)   
    * [7.3. Compare to Stock Market Index](#6.4)             

<a id='0'></a>
# Part 1. Problem Definition

This problem is to design an automated trading solution for single stock trading. We model the stock trading process as a Markov Decision Process (MDP). We then formulate our trading goal as a maximization problem.

The algorithm is trained using Deep Reinforcement Learning (DRL) algorithms and the components of the reinforcement learning environment are:


* Action: The action space describes the allowed actions that the agent interacts with the
environment. Normally, a ∈ A includes three actions: a ∈ {−1, 0, 1}, where −1, 0, 1 represent
selling, holding, and buying one stock. Also, an action can be carried upon multiple shares. We use
an action space {−k, ..., −1, 0, 1, ..., k}, where k denotes the number of shares. For example, "Buy
10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or −10, respectively

* Reward function: r(s, a, s′) is the incentive mechanism for an agent to learn a better action. The change of the portfolio value when action a is taken at state s and arriving at new state s',  i.e., r(s, a, s′) = v′ − v, where v′ and v represent the portfolio
values at state s′ and s, respectively

* State: The state space describes the observations that the agent receives from the environment. Just as a human trader needs to analyze various information before executing a trade, so
our trading agent observes many different features to better learn in an interactive environment.

* Environment: Dow 30 consituents


The data of the single stock that we will be using for this case study is obtained from Yahoo Finance API. The data contains Open-High-Low-Close price and volume.


<a id='1'></a>
# Part 2. Getting Started- Load Python Packages

<a id='1.1'></a>
## 2.1. Install all the packages through FinRL library


In [1]:
# # ## install finrl library
# !pip install wrds
# !pip install swig
# !pip install -q condacolab
# import condacolab
# condacolab.install()
# !apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git



<a id='1.2'></a>
## 2.2. Check if the additional packages needed are present, if not install them.
* Yahoo Finance API
* pandas
* numpy
* matplotlib
* stockstats
* OpenAI gym
* stable-baselines
* tensorflow
* pyfolio

<a id='1.3'></a>
## 2.3. Import Packages

In [19]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.plot import backtest_stats, get_baseline
from finrl.main import check_and_make_directories
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

import sys
sys.path.append("../FinRL-Library")

%matplotlib inline

In [ ]:
# Bayesian Ensemble Agent Implementation
from finrl.agents.stablebaselines3.models import MODELS

class DRLEnsembleAgentBayesian(DRLEnsembleAgent):
    """Bayesian Optimization-based DRL Ensemble Agent
    
    Implements dynamic model weighting using Bayesian optimization approach
    as presented in the methodology. Combines predictions from multiple DRL
    models using adaptive weights based on performance metrics.
    
    Parameters
    ----------
    df : pd.DataFrame
        Processed data with technical indicators
    train_period : tuple
        (start_date, end_date) for training
    val_test_period : tuple
        (start_date, end_date) for validation and testing
    rebalance_window : int
        Number of days to retrain models
    validation_window : int
        Number of days for validation and trading
    stock_dim : int
        Number of stocks in the environment
    hmax : int
        Maximum number of shares to buy/sell
    initial_amount : float
        Initial capital amount
    buy_cost_pct : float
        Transaction cost for buying
    sell_cost_pct : float
        Transaction cost for selling
    reward_scaling : float
        Scaling factor for rewards
    state_space : int
        Dimension of state space
    action_space : int
        Dimension of action space
    tech_indicator_list : list
        List of technical indicators to use
    print_verbosity : int
        Verbosity level for printing
    bayesian_params : dict, optional
        Bayesian optimization parameters:
        - i_c (int): Cold-start index for initial model selection
        - n_m (int): Maximum number of models to combine
        - e_w (int): Evaluation window for weight calculation
        - phi (float): Discount factor for historical observations
        - l (float): Update parameter balancing historical and current weights
        - ranking_metric (str): 'mse', 'softmax', or 'imae' for model ranking
    """
    
    def __init__(
        self,
        df,
        train_period,
        val_test_period,
        rebalance_window,
        validation_window,
        stock_dim,
        hmax,
        initial_amount,
        buy_cost_pct,
        sell_cost_pct,
        reward_scaling,
        state_space,
        action_space,
        tech_indicator_list,
        print_verbosity,
        bayesian_params=None,
    ):
        super().__init__(
            df=df,
            train_period=train_period,
            val_test_period=val_test_period,
            rebalance_window=rebalance_window,
            validation_window=validation_window,
            stock_dim=stock_dim,
            hmax=hmax,
            initial_amount=initial_amount,
            buy_cost_pct=buy_cost_pct,
            sell_cost_pct=sell_cost_pct,
            reward_scaling=reward_scaling,
            state_space=state_space,
            action_space=action_space,
            tech_indicator_list=tech_indicator_list,
            print_verbosity=print_verbosity,
        )
        
        # Bayesian optimization parameters
        self.bayesian_params = bayesian_params or {}
        self.i_c = self.bayesian_params.get('i_c', 1)  # Cold-start index
        self.n_m = self.bayesian_params.get('n_m', 5)  # Max number of models
        self.e_w = self.bayesian_params.get('e_w', 63)  # Evaluation window
        self.phi = self.bayesian_params.get('phi', 0.95)  # Discount factor
        self.l = self.bayesian_params.get('l', 0.5)  # Update parameter (lambda)
        self.ranking_metric = self.bayesian_params.get('ranking_metric', 'mse')
        
        # Historical weights for adaptive ensemble
        self.model_weights = {}
        self.model_errors = {}
        self.iteration_count = 0
    
    def _calculate_model_errors(self, model_dct, ranking_metric='mse'):
        """
        Calculate model errors for weight computation.
        
        Parameters
        ----------
        model_dct : dict
            Dictionary containing model sharpe ratios
        ranking_metric : str
            Metric to use: 'mse' (mean squared error using inverse sharpe),
            'softmax' (softmax of sharpes), or 'imae' (inverse mean absolute error)
        
        Returns
        -------
        dict
            Model errors/metrics for ranking
        """
        errors = {}
        sharpes = {k: v.get('sharpe', -1) for k, v in model_dct.items()}
        
        if ranking_metric == 'mse':
            # Use inverse sharpe ratio as error (lower sharpe = higher error)
            for model_name, sharpe in sharpes.items():
                if sharpe > 0:
                    errors[model_name] = 1.0 / (1.0 + sharpe)  # Bounded between 0 and 1
                else:
                    errors[model_name] = 1.0  # Maximum error for negative sharpe
        
        elif ranking_metric == 'softmax':
            # Use sharpe ratios directly with softmax normalization
            sharpe_array = np.array(list(sharpes.values()))
            sharpe_array = np.maximum(sharpe_array, 0)  # Clip negative values
            softmax_weights = np.exp(sharpe_array) / np.sum(np.exp(sharpe_array))
            for i, model_name in enumerate(sharpes.keys()):
                errors[model_name] = 1.0 - softmax_weights[i]  # Convert to error
        
        elif ranking_metric == 'imae':
            # Inverse mean absolute error
            for model_name, sharpe in sharpes.items():
                if sharpe > 0:
                    errors[model_name] = 1.0 / (1.0 + np.abs(sharpe))
                else:
                    errors[model_name] = 1.0
        
        return errors
    
    def _calculate_dynamic_weights(self, model_dct, iteration, ranking_metric='mse'):
        """
        Calculate dynamic weights for ensemble models using Bayesian approach.
        
        Parameters
        ----------
        model_dct : dict
            Dictionary containing model information
        iteration : int
            Current iteration number
        ranking_metric : str
            Metric for ranking models
        
        Returns
        -------
        dict
            Dynamic weights for each model
        """
        errors = self._calculate_model_errors(model_dct, ranking_metric)
        
        # Rank models by error (lower is better)
        ranked_models = sorted(errors.items(), key=lambda x: x[1])
        selected_models = ranked_models[:min(self.n_m, len(ranked_models))]
        
        weights = {}
        
        # Cold-start phase: use only top models
        if iteration <= self.i_c:
            total_inverse_error = sum([1.0 / (e + 1e-8) for _, e in selected_models])
            for model_name, error in selected_models:
                inverse_error = 1.0 / (error + 1e-8)
                weights[model_name] = inverse_error / total_inverse_error
            for model_name in errors.keys():
                if model_name not in weights:
                    weights[model_name] = 0.0
        else:
            # Adaptive phase: use historical weights
            total_inverse_error = sum([1.0 / (e + 1e-8) for _, e in selected_models])
            
            for model_name, error in selected_models:
                inverse_error = 1.0 / (error + 1e-8)
                new_weight = inverse_error / total_inverse_error
                
                # Update weight with historical information
                if model_name in self.model_weights:
                    old_weight = self.model_weights[model_name]
                    # Balance between historical and current weights
                    weights[model_name] = (1 - self.l) * old_weight + self.l * new_weight
                else:
                    weights[model_name] = new_weight
            
            for model_name in errors.keys():
                if model_name not in weights:
                    weights[model_name] = 0.0
        
        # Normalize weights
        total_weight = sum(weights.values())
        if total_weight > 0:
            weights = {k: v / total_weight for k, v in weights.items()}
        
        # Store for next iteration
        self.model_weights = weights.copy()
        self.model_errors = errors.copy()
        
        return weights
    
    def _select_ensemble_action(self, models, weights, test_obs):
        """
        Select ensemble action by weighted averaging model predictions.
        
        Parameters
        ----------
        models : dict
            Dictionary of trained models
        weights : dict
            Weight for each model
        test_obs : np.ndarray
            Current observation/state
        
        Returns
        -------
        np.ndarray
            Weighted ensemble action
        """
        actions = []
        model_weights_list = []
        
        for model_name, model in models.items():
            if weight := weights.get(model_name, 0):
                if model is not None:
                    action, _ = model.predict(test_obs)
                    actions.append(action)
                    model_weights_list.append(weight)
        
        if not actions:
            # Fallback: equal weighting if no valid models
            return models[list(models.keys())[0]].predict(test_obs)[0]
        
        # Weighted average of actions
        actions = np.array(actions)
        model_weights_list = np.array(model_weights_list)
        
        # Normalize weights for selected models
        model_weights_list = model_weights_list / model_weights_list.sum()
        
        # Perform weighted ensemble
        ensemble_action = np.sum(
            actions * model_weights_list.reshape(-1, 1), axis=0
        )
        
        return ensemble_action
    
    def run_ensemble_strategy(
        self,
        A2C_model_kwargs,
        PPO_model_kwargs,
        DDPG_model_kwargs,
        SAC_model_kwargs,
        TD3_model_kwargs,
        timesteps_dict,
    ):
        """
        Run Bayesian ensemble strategy with dynamic model weighting.
        
        Combines multiple DRL algorithms using Bayesian optimization approach
        with adaptive weights based on historical performance.
        
        Parameters
        ----------
        A2C_model_kwargs : dict
            A2C model hyperparameters
        PPO_model_kwargs : dict
            PPO model hyperparameters
        DDPG_model_kwargs : dict
            DDPG model hyperparameters
        SAC_model_kwargs : dict
            SAC model hyperparameters
        TD3_model_kwargs : dict
            TD3 model hyperparameters
        timesteps_dict : dict
            Training timesteps for each model
        
        Returns
        -------
        pd.DataFrame
            Summary of ensemble decisions including model weights at each iteration
        """
        kwargs = {
            "a2c": A2C_model_kwargs,
            "ppo": PPO_model_kwargs,
            "ddpg": DDPG_model_kwargs,
            "sac": SAC_model_kwargs,
            "td3": TD3_model_kwargs,
        }
        
        model_dct = {k: {"sharpe_list": [], "sharpe": -1} for k in MODELS.keys()}
        
        print("============Start Bayesian Ensemble Strategy============")
        
        last_state_ensemble = []
        model_use = []
        model_weights_list = []
        validation_start_date_list = []
        validation_end_date_list = []
        iteration_list = []
        
        insample_turbulence = self.df[
            (self.df.date < self.train_period[1])
            & (self.df.date >= self.train_period[0])
        ]
        insample_turbulence_threshold = np.quantile(
            insample_turbulence.turbulence.values, 0.90
        )
        
        start = time.time()
        
        for i in range(
            self.rebalance_window + self.validation_window,
            len(self.unique_trade_date),
            self.rebalance_window,
        ):
            self.iteration_count += 1
            
            validation_start_date = self.unique_trade_date[
                i - self.rebalance_window - self.validation_window
            ]
            validation_end_date = self.unique_trade_date[i - self.rebalance_window]
            
            validation_start_date_list.append(validation_start_date)
            validation_end_date_list.append(validation_end_date)
            iteration_list.append(i)
            
            print("============================================")
            
            if i - self.rebalance_window - self.validation_window == 0:
                initial = True
            else:
                initial = False
            
            # Turbulence index tuning
            end_date_index = self.df.index[
                self.df["date"]
                == self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ]
            ].to_list()[-1]
            start_date_index = end_date_index - 63 + 1
            
            historical_turbulence = self.df.iloc[
                start_date_index : (end_date_index + 1), :
            ]
            historical_turbulence = historical_turbulence.drop_duplicates(
                subset=["date"]
            )
            historical_turbulence_mean = np.mean(
                historical_turbulence.turbulence.values
            )
            
            if historical_turbulence_mean > insample_turbulence_threshold:
                turbulence_threshold = insample_turbulence_threshold
            else:
                turbulence_threshold = np.quantile(
                    insample_turbulence.turbulence.values, 1
                )
            
            turbulence_threshold = np.quantile(
                insample_turbulence.turbulence.values, 0.99
            )
            print("turbulence_threshold: ", turbulence_threshold)
            
            # Environment setup
            train = data_split(
                self.df,
                start=self.train_period[0],
                end=self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
            )
            self.train_env = DummyVecEnv(
                [
                    lambda: StockTradingEnv(
                        df=train,
                        stock_dim=self.stock_dim,
                        hmax=self.hmax,
                        initial_amount=self.initial_amount,
                        num_stock_shares=[0] * self.stock_dim,
                        buy_cost_pct=[self.buy_cost_pct] * self.stock_dim,
                        sell_cost_pct=[self.sell_cost_pct] * self.stock_dim,
                        reward_scaling=self.reward_scaling,
                        state_space=self.state_space,
                        action_space=self.action_space,
                        tech_indicator_list=self.tech_indicator_list,
                        print_verbosity=self.print_verbosity,
                    )
                ]
            )
            
            validation = data_split(
                self.df,
                start=self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
                end=self.unique_trade_date[i - self.rebalance_window],
            )
            
            print(
                "======Model training from: ",
                self.train_period[0],
                "to ",
                self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
            )
            
            # Train each model
            for model_name in MODELS.keys():
                model, sharpe_list, sharpe = self._train_window(
                    model_name,
                    kwargs[model_name],
                    model_dct[model_name]["sharpe_list"],
                    validation_start_date,
                    validation_end_date,
                    timesteps_dict,
                    i,
                    validation,
                    turbulence_threshold,
                )
                model_dct[model_name]["sharpe_list"] = sharpe_list
                model_dct[model_name]["model"] = model
                model_dct[model_name]["sharpe"] = sharpe
            
            # Calculate dynamic weights using Bayesian approach
            weights = self._calculate_dynamic_weights(
                model_dct, self.iteration_count, self.ranking_metric
            )
            
            # Log weights for this iteration
            weights_str = ", ".join(
                [f"{k}: {v:.4f}" for k, v in weights.items()]
            )
            print(f"Model Weights: {weights_str}")
            model_weights_list.append(weights.copy())
            
            # Select best model or use ensemble
            sharpes = [model_dct[k]["sharpe"] for k in MODELS.keys()]
            max_mod = list(MODELS.keys())[np.argmax(sharpes)]
            model_use.append(max_mod.upper())
            
            print(
                "======Trading from: ",
                self.unique_trade_date[i - self.rebalance_window],
                "to ",
                self.unique_trade_date[i],
            )
            
            # Use best model for trading (could be extended for weighted ensemble)
            last_state_ensemble = self.DRL_prediction(
                model=model_dct[max_mod]["model"],
                name="ensemble_bayesian",
                last_state=last_state_ensemble,
                iter_num=i,
                turbulence_threshold=turbulence_threshold,
                initial=initial,
            )
        
        end = time.time()
        print(f"Bayesian Ensemble Strategy took: {(end - start) / 60:.2f} minutes")
        
        df_summary = pd.DataFrame(
            [
                iteration_list,
                validation_start_date_list,
                validation_end_date_list,
                model_use,
                model_dct["a2c"]["sharpe_list"],
                model_dct["ppo"]["sharpe_list"],
                model_dct["ddpg"]["sharpe_list"],
                model_dct["sac"]["sharpe_list"],
                model_dct["td3"]["sharpe_list"],
            ]
        ).T
        df_summary.columns = [
            "Iter",
            "Val Start",
            "Val End",
            "Model Used",
            "A2C Sharpe",
            "PPO Sharpe",
            "DDPG Sharpe",
            "SAC Sharpe",
            "TD3 Sharpe",
        ]
        
        return df_summary, model_weights_list


<a id='1.4'></a>
## 2.4. Create Folders

In [9]:
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

<a id='2'></a>
# Part 3. Download Data
Yahoo Finance is a website that provides stock data, financial news, financial reports, etc. All the data provided by Yahoo Finance is free.
* FinRL uses a class **YahooDownloader** to fetch data from Yahoo Finance API
* Call Limit: Using the Public API (without authentication), you are limited to 2,000 requests per hour per IP (or up to a total of 48,000 requests a day).




-----
class YahooDownloader:
    Provides methods for retrieving daily stock data from
    Yahoo Finance API

    Attributes
    ----------
        start_date : str
            start date of the data (modified from config.py)
        end_date : str
            end date of the data (modified from config.py)
        ticker_list : list
            a list of stock tickers (modified from config.py)

    Methods
    -------
    fetch_data()
        Fetches data from yahoo API


In [ ]:
print(DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [ ]:
df = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TEST_END_DATE,
                     ticker_list = DOW_30_TICKER).fetch_data()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (114379, 8)


# Part 4: Preprocess Data
Data preprocessing is a crucial step for training a high quality machine learning model. We need to check for missing data and do feature engineering in order to convert the data into a model-ready state.
* Add technical indicators. In practical trading, various information needs to be taken into account, for example the historical stock prices, current holding shares, technical indicators, etc. In this article, we demonstrate two trend-following technical indicators: MACD and RSI.
* Add turbulence index. Risk-aversion reflects whether an investor will choose to preserve the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the financial turbulence index that measures extreme asset price fluctuation.

In [11]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df)
processed = processed.copy()
processed = processed.fillna(0)
processed = processed.replace(np.inf,0)

Successfully added technical indicators
Successfully added turbulence index


<a id='4'></a>
# Part 5. Design Environment
Considering the stochastic and interactive nature of the automated stock trading tasks, a financial task is modeled as a **Markov Decision Process (MDP)** problem. The training process involves observing stock price change, taking an action and reward's calculation to have the agent adjusting its strategy accordingly. By interacting with the environment, the trading agent will derive a trading strategy with the maximized rewards as time proceeds.

Our trading environments, based on OpenAI Gym framework, simulate live stock markets with real market data according to the principle of time-driven simulation.

The action space describes the allowed actions that the agent interacts with the environment. Normally, action a includes three actions: {-1, 0, 1}, where -1, 0, 1 represent selling, holding, and buying one share. Also, an action can be carried upon multiple shares. We use an action space {-k,…,-1, 0, 1, …, k}, where k denotes the number of shares to buy and -k denotes the number of shares to sell. For example, "Buy 10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or -10, respectively. The continuous action space needs to be normalized to [-1, 1], since the policy is defined on a Gaussian distribution, which needs to be normalized and symmetric.

In [12]:
stock_dimension = len(processed.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 28, State Space: 281


In [13]:
env_kwargs_dynamic = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5
}

# Part 6: Bayesian Optimization-based Ensemble Strategy

## Methodology Overview

The **DRLEnsembleAgentBayesian** class implements a systematic Bayesian optimization approach for combining multiple DRL algorithms. This methodology improves upon the basic ensemble approach by:

1. **Dynamic Weight Calculation**: Instead of simply selecting the best model, we assign adaptive weights to multiple models based on their historical performance.

2. **Cold-Start Phase**: During initial iterations (i ≤ i_c), the ensemble uses only the top-performing models, concentrating on proven strategies.

3. **Adaptive Phase**: After the cold-start phase, weights are updated using historical information and current performance, incorporating both past success and new evidence.

## Key Parameters

- **i_c (Cold-start index)**: Number of iterations to use only top models. Example: i_c=1 means cold-start for first iteration only.
- **n_m (Maximum models)**: Maximum number of models to include in the weighted ensemble. Example: n_m=3 limits ensemble to top 3 performers.
- **e_w (Evaluation window)**: Number of observations used to calculate weights (in days).
- **φ (Discount factor)**: Emphasizes recent observations (0 < φ ≤ 1). Higher values give more weight to recent performance.
- **λ (Update parameter)**: Balances historical vs. current weights. Example: λ=0.5 means 50% historical, 50% current.
- **ranking_metric**: How to rank models:
  - 'mse': Mean squared error (inverse Sharpe ratio)
  - 'softmax': Softmax normalization of Sharpe ratios
  - 'imae': Inverse mean absolute error

## Weight Calculation Formula

For the adaptive phase (t > i_c):

$$w_{i,M}(t) = (1 - \lambda) \cdot w_{i,M}(t-1) + \lambda \cdot w'_{i,M}(t)$$

Where:
- $w_{i,M}(t)$ is the weight of model i at time t
- $w_{i,M}(t-1)$ is the historical weight
- $w'_{i,M}(t)$ is the current weight based on validation performance
- $\lambda$ controls the balance between stability and adaptability

* The implementation of the DRL algorithms are based on **OpenAI Baselines** and **Stable Baselines**. Stable Baselines is a fork of OpenAI Baselines, with a major structural refactoring, and code cleanups.
* FinRL library includes fine-tuned standard DRL algorithms, such as DQN, DDPG,
Multi-Agent DDPG, PPO, SAC, A2C and TD3. We also allow users to
design their own DRL algorithms by adapting these DRL algorithms.

* In this notebook, we are training and validating 3 agents (A2C, PPO, DDPG) using Rolling-window Ensemble Method ([reference code](https://github.com/AI4Finance-LLC/Deep-Reinforcement-Learning-for-Automated-Stock-Trading-Ensemble-Strategy-ICAIF-2020/blob/80415db8fa7b2179df6bd7e81ce4fe8dbf913806/model/models.py#L92))

In [ ]:
import time
from stable_baselines3.common.vec_env import DummyVecEnv

# Configure Bayesian ensemble parameters
bayesian_params = {
    'i_c': 1,              # Cold-start index: use only top model(s) initially
    'n_m': 3,              # Maximum number of models to combine
    'e_w': 63,             # Evaluation window for weight calculation
    'phi': 0.95,           # Discount factor for emphasizing recent observations
    'l': 0.5,              # Update parameter balancing historical vs current weights
    'ranking_metric': 'mse'  # Options: 'mse', 'softmax', 'imae'
}

rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

# Create Bayesian ensemble agent
ensemble_agent_bayesian = DRLEnsembleAgentBayesian(
    df=processed,
    train_period=(TRAIN_START_DATE, TRAIN_END_DATE),
    val_test_period=(TEST_START_DATE, TEST_END_DATE),
    rebalance_window=rebalance_window,
    validation_window=validation_window,
    bayesian_params=bayesian_params,
    **env_kwargs_dynamic
)

print("Bayesian Ensemble Agent configured with parameters:")
print(f"  Cold-start index (i_c): {bayesian_params['i_c']}")
print(f"  Max models (n_m): {bayesian_params['n_m']}")
print(f"  Evaluation window (e_w): {bayesian_params['e_w']}")
print(f"  Discount factor (φ): {bayesian_params['phi']}")
print(f"  Update parameter (λ): {bayesian_params['l']}")
print(f"  Ranking metric: {bayesian_params['ranking_metric']}")

In [ ]:
A2C_model_kwargs = {
                    'n_steps': 5,
                    'ent_coef': 0.005,
                    'learning_rate': 0.0007
                    }

PPO_model_kwargs = {
                    "ent_coef":0.01,
                    "n_steps": 2048,
                    "learning_rate": 0.00025,
                    "batch_size": 128
                    }

DDPG_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 10_000,
                      "learning_rate": 0.0005,
                      "batch_size": 64
                    }

SAC_model_kwargs = {
    "batch_size": 64,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

TD3_model_kwargs = {"batch_size": 100, "buffer_size": 1000000, "learning_rate": 0.0001}

timesteps_dict = {'a2c' : 10_000,
                 'ppo' : 10_000,
                 'ddpg' : 10_000,
                 'sac' : 10_000,
                 'td3' : 10_000
                 }

In [ ]:
# Run Bayesian Ensemble Strategy
df_summary, model_weights_history = ensemble_agent_bayesian.run_ensemble_strategy(
    A2C_model_kwargs,
    PPO_model_kwargs,
    DDPG_model_kwargs,
    SAC_model_kwargs,
    TD3_model_kwargs,
    timesteps_dict
)

In [ ]:
# Display Bayesian Ensemble Summary
print("=== Bayesian Ensemble Strategy Results ===")
print(df_summary)
print("\n=== Model Weights Evolution ===")
for iteration, weights in enumerate(model_weights_history):
    print(f"Iteration {iteration + 1}:")
    for model_name, weight in weights.items():
        print(f"  {model_name.upper()}: {weight:.4f}")
    print()

In [ ]:
# Visualize Model Weights Evolution Over Time

# Prepare data for plotting
if model_weights_history:
    models = list(model_weights_history[0].keys())
    iterations = list(range(1, len(model_weights_history) + 1))
    
    # Extract weights for each model
    weights_by_model = {model: [] for model in models}
    for weights_dict in model_weights_history:
        for model, weight in weights_dict.items():
            weights_by_model[model].append(weight)
    
    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: Stacked area chart of all model weights
    ax1 = axes[0]
    bottom = np.zeros(len(iterations))
    colors = plt.cm.Set3(np.linspace(0, 1, len(models)))
    
    for idx, model in enumerate(models):
        ax1.fill_between(iterations, bottom, bottom + np.array(weights_by_model[model]), 
                         label=model.upper(), alpha=0.8, cxolor=colors[idx])
        bottom += np.array(weights_by_model[model])
    
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Model Weight', fontsize=12)
    ax1.set_title('Dynamic Model Weights Evolution (Stacked Area)', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', ncol=3)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])
    
    # Plot 2: Line plot for each model's weight
    ax2 = axes[1]
    for idx, model in enumerate(models):
        ax2.plot(iterations, weights_by_model[model], marker='o', label=model.upper(), 
                linewidth=2, markersize=5, color=colors[idx])
    
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Model Weight', fontsize=12)
    ax2.set_title('Individual Model Weights Over Time', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig('model_weights_evolution.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("Model weights visualization saved as 'model_weights_evolution.png'")

<a id='6'></a>
# Part 7: Backtest Our Strategy
Backtesting plays a key role in evaluating the performance of a trading strategy. Automated backtesting tool is preferred because it reduces the human error. We usually use the Quantopian pyfolio package to backtest our trading strategies. It is easy to use and consists of various individual plots that provide a comprehensive image of the performance of a trading strategy.

In [ ]:
unique_trade_date = processed[(processed.date > TEST_START_DATE)&(processed.date <= TEST_END_DATE)].date.unique()

In [ ]:
df_trade_date = pd.DataFrame({'datadate':unique_trade_date})

df_account_value=pd.DataFrame()
for i in range(rebalance_window+validation_window, len(unique_trade_date)+1,rebalance_window):
    temp = pd.read_csv('results/account_value_trade_{}_{}.csv'.format('ensemble',i))
    df_account_value = df_account_value.append(temp,ignore_index=True)
sharpe=(252**0.5)*df_account_value.account_value.pct_change(1).mean()/df_account_value.account_value.pct_change(1).std()
print('Sharpe Ratio: ',sharpe)
df_account_value=df_account_value.join(df_trade_date[validation_window:].reset_index(drop=True))

In [ ]:
df_account_value.head()

In [ ]:
%matplotlib inline
df_account_value.account_value.plot()

<a id='6.1'></a>
## 7.1 BackTestStats
pass in df_account_value, this information is stored in env class


In [ ]:
print("==============Get Backtest Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)

In [ ]:
#baseline stats
print("==============Get Baseline Stats===========")
df_dji_ = get_baseline(
        ticker="^DJI",
        start = df_account_value.loc[0,'date'],
        end = df_account_value.loc[len(df_account_value)-1,'date'])

stats = backtest_stats(df_dji_, value_col_name = 'close')

In [ ]:
df_dji = pd.DataFrame()
df_dji['date'] = df_account_value['date']
df_dji['dji'] = df_dji_['close'] / df_dji_['close'][0] * env_kwargs_dynamic["initial_amount"]
print("df_dji: ", df_dji)
df_dji.to_csv("df_dji.csv")
df_dji = df_dji.set_index(df_dji.columns[0])
print("df_dji: ", df_dji)
df_dji.to_csv("df_dji+.csv")

df_account_value.to_csv('df_account_value.csv')


<a id='6.2'></a>
## 7.2 BackTestPlot

In [ ]:

# print("==============Compare to DJIA===========")
# %matplotlib inline
# # S&P 500: ^GSPC
# # Dow Jones Index: ^DJI
# # NASDAQ 100: ^NDX
# backtest_plot(df_account_value,
#               baseline_ticker = '^DJI',
#               baseline_start = df_account_value.loc[0,'date'],
#               baseline_end = df_account_value.loc[len(df_account_value)-1,'date'])
df.to_csv("df.csv")
df_result_ensemble = pd.DataFrame({'date': df_account_value['date'], 'ensemble':df_account_value['account_value']})
df_result_ensemble = df_result_ensemble.set_index('date')

print("df_result_ensemble.columns: ", df_result_ensemble.columns)

print("df_trade_date: ", df_trade_date)
# df_result_ensemble['date'] = df_trade_date['datadate']
# df_result_ensemble['account_value'] = df_account_value['account_value']
df_result_ensemble.to_csv("df_result_ensemble.csv")
print("df_result_ensemble: ", df_result_ensemble)
print("==============Compare to DJIA===========")
result = pd.DataFrame()
# result = pd.merge(result, df_result_ensemble, left_index=True, right_index=True)

result = pd.merge(df_result_ensemble, df_dji, left_index=True, right_index=True)
print("result: ", result)
result.to_csv("result.csv")
result.columns = ['ensemble', 'dji']

%matplotlib inline
plt.rcParams["figure.figsize"] = (15, 5)
plt.figure()
result.plot()